# 🤖 Emotion AI Chatbot

In this project, we build an intelligent chatbot that can understand user emotions from text and  images and respond in a natural, empathetic way.

The system uses a trained emotion classification model to detect the user's emotional state, applies simple logic to refine the prediction, and then generates context-aware responses. The goal is to create a more human-like conversational experience by combining machine learning with rule-based reasoning.

This project demonstrates the integration of NLP, computer vision , and chatbot design, and can be deployed  Gradio.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

#Text Dataset


In [ ]:
from datasets import load_dataset

ds = load_dataset("vbhaskar/go_emotions", "simplified")
print("data loaded sucessfully")

In [ ]:
print(ds)

In [ ]:
ds['train'].features

In [ ]:
df = ds["train"].to_pandas()
df.head()

#Exploratory Data Analysis

In [ ]:
df.info()

# 📊 Text Emotion Dataset

The dataset used in this project is a multi-label emotion classification dataset consisting of textual data annotated with emotional categories.

It is divided into three splits:

* **Training set**: 43,410 samples
* **Validation set**: 5,426 samples
* **Test set**: 5,427 samples

Each data sample contains:

* **text**: the input sentence or user message
* **labels**: one or more associated emotions
* **id**: a unique identifier

The dataset includes **28 emotion classes**, such as:
*joy, sadness, anger, fear, love, surprise, neutral,* and others.

Since each text can have multiple emotions, this is a **multi-label classification problem**, making it more complex and realistic compared to single-label emotion detection.

The dataset is converted into a pandas DataFrame for preprocessing, tokenization, and model training.


In [ ]:
num_labels = 28

def encode_labels(label_list):
    multi_hot = [0] * num_labels
    for label in label_list:
        multi_hot[label] = 1
    return multi_hot

df["labels"] = df["labels"].apply(encode_labels)

In [ ]:
df["labels"].head()

In [ ]:
label_counts = np.sum(df["labels"].tolist(), axis=0)

In [ ]:
emotion_labels = [
    "admiration","amusement","anger","annoyance","approval","caring","confusion",
    "curiosity","desire","disappointment","disapproval","disgust","embarrassment",
    "excitement","fear","gratitude","grief","joy","love","nervousness","optimism",
    "pride","realization","relief","remorse","sadness","surprise","neutral"
]

plt.figure(figsize=(14,6))
plt.bar(emotion_labels, label_counts)

plt.xticks(rotation=90)
plt.title("Emotion Distribution")
plt.xlabel("Emotion")
plt.ylabel("Count")

plt.show()

In [ ]:
#Text length
df["text_length"] = df["text"].apply(lambda x: len(x.split()))

In [ ]:
plt.hist(df["text_length"], bins=50)
plt.title("Text Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
from collections import Counter

all_words = " ".join(df["text"]).split()
word_counts = Counter(all_words)

print(word_counts.most_common(20))

In [ ]:
from wordcloud import WordCloud

text = " ".join(df["text"])

wordcloud = WordCloud(width=800, height=400).generate(text)

plt.imshow(wordcloud)
plt.axis("off")
plt.show()

In [ ]:
df["num_labels"] = df["labels"].apply(lambda x: sum(x))

In [ ]:
plt.hist(df["num_labels"], bins=10)
plt.title("Number of Emotions per Text")
plt.xlabel("Count")
plt.show()

#Preprocessing

In [ ]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   # remove links
    text = re.sub(r"@\w+", "", text)      # remove mentions
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove special chars
    return text

# 🧹 Text Preprocessing

Before training the model, the raw text data is cleaned to remove noise and make it suitable for learning.

The preprocessing function performs the following steps:

1. **Lowercasing**
   Converts all text to lowercase to ensure uniformity (e.g., "Happy" → "happy").

2. **Remove URLs**
   Any web links (e.g., http://...) are removed since they do not contribute to emotion detection.

3. **Remove Mentions**
   User mentions (e.g., @username) are removed as they are irrelevant to the emotional content.

4. **Remove Special Characters**
   All non-alphabetic characters (numbers, punctuation, symbols) are removed, keeping only letters and spaces.

This helps:

* Reduce noise in the data
* Improve model performance
* Ensure consistent input format

The cleaned text is then used for tokenization and training the emotion classification model.


In [ ]:
num_labels = 28

def encode_labels(example):
    multi_hot = [0] * num_labels
    for label in example["labels"]:
        multi_hot[label] = 1
    example["labels"] = multi_hot
    return example

ds = ds.map(encode_labels)

In [ ]:
print(ds["train"][0]["labels"])
print(len(ds["train"][0]["labels"]))

In [ ]:
ds = ds.map(lambda x: {"text": clean_text(x["text"])})
print('Mapped')

In [ ]:
ds = ds.remove_columns("id")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

ds = ds.map(tokenize, batched=True)

In [ ]:
import torch

ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    format_kwargs={"dtype": torch.float}
)

In [ ]:
print(len(ds["train"][0]["labels"]))
print(ds["train"][0]["labels"].dtype)

In [ ]:
ds = ds.map(lambda x: {"labels": [float(i) for i in x["labels"]]})

In [ ]:
ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
print(type(ds["train"][0]["labels"][0]))

In [ ]:
ds["train"][0]["labels"].dtype

In [ ]:
ds = ds.map(lambda x: {"labels": [float(i) for i in x["labels"]]})

In [ ]:
ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
ds["train"][0]

In [ ]:
ds["train"][0]["labels"].dtype

In [ ]:
from datasets import Features, Sequence, Value

new_features = Features({
    "input_ids": Sequence(Value("int64")),
    "attention_mask": Sequence(Value("int64")),
    "labels": Sequence(Value("float32")),
})

In [ ]:
ds = ds.remove_columns(["text", "token_type_ids"])

In [ ]:
from datasets import Features, Sequence, Value

new_features = Features({
    "input_ids": Sequence(Value("int64")),
    "attention_mask": Sequence(Value("int64")),
    "labels": Sequence(Value("float32")),
})

ds = ds.cast(new_features)

In [ ]:
ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
ds["train"][0]["labels"].dtype

In [ ]:
df = ds["train"].to_pandas()
df.head()

input_ids
 This is the numerical version of  sentence

token_type_ids
Used to distinguish sentence pairs

attention_mask
Tells model what to focus on

#BERT

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.metrics import f1_score

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=28,
    problem_type="multi_label_classification"
)

DistilBERT (distilbert-base-uncased) is a lightweight transformer model used for text classification. It is faster and smaller than BERT while still maintaining strong performance in understanding language context. The model is configured with 28 labels and a multi-label classification setup, allowing it to detect multiple emotions from a single text. It was chosen because it provides a good balance between accuracy and efficiency, making it suitable for real-time applications like chatbots.


In [ ]:

model.to("cuda")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)

    f1 = f1_score(labels, preds, average="micro")
    return {"f1": f1}

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

The TrainingArguments define how the model is trained, including learning rate, batch size, number of epochs, and evaluation strategy. The model is trained for 3 epochs with a learning rate of 2e-5 and batch size of 8, balancing performance and computational efficiency. The Trainer API from Hugging Face simplifies the training process by handling training, evaluation, and logging automatically. It uses the training and validation datasets along with a custom metric function to monitor model performance.


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict_emotion(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    return probs

In [ ]:
def get_top_emotions(probs, top_k=3):
    top_indices = probs.argsort()[-top_k:][::-1]
    return [(emotion_labels[i], float(probs[i])) for i in top_indices]

In [ ]:
def get_emotions(probs, threshold=0.2):
    emotions = []

    for i, p in enumerate(probs):
        if p > threshold:
            emotions.append((emotion_labels[i], float(p)))

    if len(emotions) == 0:
        top_indices = probs.argsort()[-3:][::-1]
        emotions = [(emotion_labels[i], float(probs[i])) for i in top_indices]

    return emotions

In [ ]:
def interpret_emotion(emotions):
    positive = ["joy", "optimism", "admiration", "approval", "gratitude","love"]
    negative = ["sadness", "grief"]

    for emo, score in emotions:
        if emo in positive:
            return "Positive emotional state"
        elif emo in negative:
            return "Negative emotional state"

    return "Neutral or mixed emotions"

In [ ]:

# text=input("Enter your sentence:")
# probs=predict_emotion(text)
# emotions=get_emotions(probs)
# print("\nDetected Emotions:")
# for emo,score in emotions:
#   print(f"{emo}:{score:.2f}")
# interpretation=interpret_emotion(emotions)
# print("\nInterpretation:",interpretation)

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(ds["test"])
logits = predictions.predictions
true_labels = predictions.label_ids

In [ ]:
probs = 1 / (1 + np.exp(-logits))  # sigmoid

In [ ]:
threshold = 0.3
preds = (probs > threshold).astype(int)

The model outputs raw scores (logits), which are converted into probabilities using the sigmoid function since this is a multi-label classification task. A threshold of 0.3 is then applied to these probabilities to decide which emotions are present. If a probability exceeds the threshold, it is marked as 1 (emotion present), otherwise 0. This allows the model to predict multiple emotions for a single input instead of just one.


In [ ]:
print(classification_report(true_labels, preds, target_names=emotion_labels))

The model shows moderate performance with a micro F1-score of 0.60, indicating a balanced overall prediction across all labels. It performs well on common emotions like *gratitude, love,* and *amusement*, which have high precision and recall. However, performance is lower for rare classes such as *grief* and *relief*, mainly due to limited training samples. Overall, the model is effective for general emotion detection but can be improved for less frequent emotions.


In [ ]:
trainer.save_model("my_emotion_model")

In [ ]:
tokenizer.save_pretrained("my_emotion_model")

#Image Dataset

In [ ]:
import os
import kagglehub
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d msambare/fer2013

In [ ]:
!unzip fer2013.zip

In [ ]:
import os
os.listdir()

In [ ]:
train_path = "train"

for folder in os.listdir(train_path):
    print(folder)

In [ ]:
class_counts = {}

for cls in os.listdir(train_path):
    class_counts[cls] = len(os.listdir(os.path.join(train_path, cls)))

print(class_counts)

In [ ]:

plt.figure(figsize=(8,5))
plt.bar(class_counts.keys(), class_counts.values())
plt.title("Class Distribution (FER2013)")
plt.xlabel("Emotion")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
from PIL import Image
import random

plt.figure(figsize=(12,8))

for i, cls in enumerate(class_counts.keys()):
    img_path = os.path.join(train_path, cls, random.choice(os.listdir(os.path.join(train_path, cls))))
    img = Image.open(img_path)

    plt.subplot(2,4,i+1)
    plt.imshow(img, cmap='gray')
    plt.title(cls)
    plt.axis("off")

plt.show()

# 🖼 Image Emotion Dataset

The image dataset used in this project is based on the FER-2013 dataset, which contains facial expressions categorized into different emotions.

The dataset is organized into separate folders such as **train** and **test**, with images grouped into emotion classes like *angry, happy, fear, neutral, sad, surprise,* and *disgust*.

The distribution of some classes includes:

* angry: 3995 images
* happy: 7215 images
* fear: 4097 images
* neutral: 4965 images
* sad: 4830 images

This dataset is slightly imbalanced, with some emotions like *happy* having more samples than others. The images are used to train a deep learning model to recognize facial expressions and predict the corresponding emotion.


In [ ]:
img = Image.open(img_path)
print(img.size)

In [ ]:
pixels = []

for cls in os.listdir(train_path):
    for img_name in os.listdir(os.path.join(train_path, cls))[:100]:
        img = Image.open(os.path.join(train_path, cls, img_name))
        pixels.extend(np.array(img).flatten())

plt.hist(pixels, bins=50)
plt.title("Pixel Distribution")
plt.show()

In [ ]:
sorted_counts = dict(sorted(class_counts.items(), key=lambda x: x[1], reverse=True))
print(sorted_counts)

#Preprocessing


In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # ensure 1 channel
    transforms.Resize((48, 48)),                 # same size
    transforms.ToTensor(),                       # convert to tensor (0–1)
])

In [ ]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [ ]:
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((48, 48)),

    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15)),

    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader

train_data = datasets.ImageFolder("train", transform=train_transform)
test_data = datasets.ImageFolder("test", transform=test_transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

In [ ]:
import torch
from sklearn.utils.class_weight import compute_class_weight

labels = [label for _, label in train_data]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

#CNN

In [ ]:
# import torch.nn as nn
# import torch.nn.functional as F

# class EmotionCNN(nn.Module):
#     def __init__(self):
#         super(EmotionCNN, self).__init__()

#         self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
#         self.bn1 = nn.BatchNorm2d(32)

#         self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
#         self.bn2 = nn.BatchNorm2d(64)

#         self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
#         self.bn3 = nn.BatchNorm2d(128)

#         self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
#         self.bn4 = nn.BatchNorm2d(256)

#         self.pool = nn.MaxPool2d(2, 2)
#         self.dropout = nn.Dropout(0.3)  # reduced dropout

#         self.fc1 = nn.Linear(256 * 3 * 3, 512)
#         self.fc2 = nn.Linear(512, 7)

#     def forward(self, x):
#         x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 48 → 24
#         x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 24 → 12
#         x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 12 → 6
#         x = self.pool(F.relu(self.bn4(self.conv4(x))))  # 6 → 3

#         x = x.view(x.size(0), -1)

#         x = self.dropout(F.relu(self.fc1(x)))
#         x = self.fc2(x)

#         return x

In [ ]:
# import torch.optim as optim

# model = EmotionCNN().to(device)

# criterion = nn.CrossEntropyLoss(weight=class_weights)
# optimizer = optim.Adam(model.parameters(), lr=0.0003)


# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode='max', patience=2, factor=0.5
# )

In [ ]:
# num_epochs = 30

# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0

#     for images, labels in train_loader:
#         images, labels = images.to(device), labels.to(device)

#         optimizer.zero_grad()

#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item() * images.size(0)

#     epoch_loss = running_loss / len(train_loader.dataset)

#     # Evaluation
#     model.eval()
#     correct = 0
#     total = 0

#     with torch.no_grad():
#         for images, labels in test_loader:
#             images, labels = images.to(device), labels.to(device)

#             outputs = model(images)
#             _, predicted = torch.max(outputs, 1)

#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()

#     acc = 100 * correct / total

#     scheduler.step(acc)

#     print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}, Accuracy: {acc:.2f}%")

In [ ]:
# best_acc = 0

# if acc > best_acc:
#     best_acc = acc
#     torch.save(model.state_dict(), "best_model.pth")

In [ ]:
# model.eval()

# y_pred = []
# y_true = []

# with torch.no_grad():
#     for images, labels in test_loader:
#         images = images.to(device)

#         outputs = model(images)
#         _, predicted = torch.max(outputs, 1)

#         y_pred.extend(predicted.cpu().numpy())
#         y_true.extend(labels.numpy())

In [ ]:
# y_pred = np.array(y_pred)
# y_true = np.array(y_true)

# accuracy = (y_pred == y_true).mean() * 100
# print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
# from sklearn.metrics import confusion_matrix
# import seaborn as sns
# import matplotlib.pyplot as plt

# cm = confusion_matrix(y_true, y_pred)

# plt.figure(figsize=(8,6))
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
#             xticklabels=train_data.classes,
#             yticklabels=train_data.classes)

# plt.xlabel("Predicted")
# plt.ylabel("Actual")
# plt.title("Confusion Matrix")
# plt.show()

In [ ]:
# from sklearn.metrics import classification_report

# print(classification_report(y_true, y_pred, target_names=train_data.classes))

In [ ]:
# !pip install torchsummary

In [ ]:
# from torchsummary import summary

# summary(model, (1, 48, 48))

#ResNet18

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),   # improves accuracy
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(root="train", transform=transform)
val_dataset   = datasets.ImageFolder(root="test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model_res = models.resnet18(pretrained=True)

ResNet18 is a convolutional neural network that uses residual (skip) connections to allow deeper learning without vanishing gradient problems. It is lightweight and faster compared to larger models, making it suitable for training in limited environments like Colab. I chose ResNet18 because it provides a good balance between accuracy and computational efficiency for image classification tasks. It is also widely used and pretrained weights are available, which helps improve performance even with smaller datasets.


In [ ]:
num_classes = len(train_dataset.classes)

model_res.fc = nn.Linear(model_res.fc.in_features, num_classes)
model_res = model_res.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_res.parameters(), lr=0.0001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model_res.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model_res(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss:.4f}, Accuracy: {train_acc:.2f}%")

    # Validation
    model_res.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model_res(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f"Validation Accuracy: {val_acc:.2f}%")

The model is trained for 10 epochs using a training loop that updates weights through backpropagation and optimization. During each epoch, the model processes training images, calculates loss, and improves its predictions while tracking training accuracy. After each epoch, the model is evaluated on the validation dataset without updating weights to measure its generalization performance. This helps monitor both training and validation accuracy to ensure the model is learning effectively and not overfitting.


In [ ]:
torch.save(model_res.state_dict(), "resnet_model.pth")

#Combined Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path = "/content/drive/MyDrive/emotion_app"

os.makedirs(f"{base_path}/text_model", exist_ok=True)
os.makedirs(f"{base_path}/image_model", exist_ok=True)
os.makedirs(f"{base_path}/labels", exist_ok=True)

print("Folders created!")

In [ ]:
#save text model
trainer.save_model(f"{base_path}/text_model")
tokenizer.save_pretrained(f"{base_path}/text_model")

In [ ]:
#Image model
torch.save(model_res.state_dict(), f"{base_path}/image_model/resnet_model.pth")

In [ ]:
#Save Labels
import json

text_class_names = [
    'admiration','amusement','anger','annoyance','approval','caring','confusion',
    'curiosity','desire','disappointment','disapproval','disgust','embarrassment',
    'excitement','fear','gratitude','grief','joy','love','nervousness','optimism',
    'pride','realization','relief','remorse','sadness','surprise','neutral'
]

image_class_names = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

with open(f"{base_path}/labels/text_labels.json", "w") as f:
    json.dump(text_class_names, f)

with open(f"{base_path}/labels/image_labels.json", "w") as f:
    json.dump(image_class_names, f)

print("Labels saved!")

In [ ]:
#combned model class
import torch
import json
import cv2
from PIL import Image
from torchvision import transforms, models
from transformers import AutoTokenizer, AutoModelForSequenceClassification


class EmotionAI:

    def __init__(self, base_path, device=None):

        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")

        # Load labels
        with open(f"{base_path}/labels/text_labels.json") as f:
            self.text_labels = json.load(f)

        with open(f"{base_path}/labels/image_labels.json") as f:
            self.image_labels = json.load(f)

        # Load text model
        self.tokenizer = AutoTokenizer.from_pretrained(f"{base_path}/text_model")
        self.text_model = AutoModelForSequenceClassification.from_pretrained(
            f"{base_path}/text_model"
        ).to(self.device)

        self.text_model.eval()

        # Load image model
        self.image_model = models.resnet18()
        self.image_model.fc = torch.nn.Linear(
            self.image_model.fc.in_features, len(self.image_labels)
        )

        self.image_model.load_state_dict(
            torch.load(f"{base_path}/image_model/resnet_model.pth", map_location=self.device)
        )

        self.image_model.to(self.device)
        self.image_model.eval()

        # Transform
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])

        # Face detector
        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )

    def map_to_fer(self, label):
        if label in ['anger', 'annoyance']:
            return 'angry'
        elif label in ['disgust']:
            return 'disgust'
        elif label in ['fear', 'nervousness']:
            return 'fear'
        elif label in ['joy', 'amusement', 'love', 'excitement', 'optimism']:
            return 'happy'
        elif label in ['sadness', 'grief', 'disappointment', 'remorse']:
            return 'sad'
        elif label in ['surprise', 'realization']:
            return 'surprise'
        else:
            return 'neutral'

    def predict_text(self, text):

        inputs = self.tokenizer(
            text, return_tensors="pt", truncation=True, padding=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.text_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)[0]

        topk = torch.topk(probs, 5)

        seen = set()
        results = []

        for idx in topk.indices:
            label = self.text_labels[idx]
            confidence = probs[idx].item()

            mapped = self.map_to_fer(label)

            if mapped not in seen:
                seen.add(mapped)
                results.append((mapped, confidence))

            if len(results) == 2:
                break

        return results

    def predict_image(self, image_path):

        image_cv = cv2.imread(image_path)
        gray = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)

        faces = self.face_cascade.detectMultiScale(gray, 1.3, 5)

        if len(faces) > 0:
            (x, y, w, h) = faces[0]
            face = image_cv[y:y+h, x:x+w]
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(face)
        else:
            image = Image.open(image_path).convert("RGB")

        image = self.transform(image).unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.image_model(image)
            probs = torch.softmax(outputs, dim=1)[0]

        top2 = torch.topk(probs, 2)

        results = []
        for idx in top2.indices:
            label = self.image_labels[idx]
            confidence = probs[idx].item()
            results.append((label, confidence))

        return results

    def predict(self, user_input):

        if user_input.lower().endswith(('.jpg', '.png', '.jpeg')):
            return self.predict_image(user_input)
        else:
            return self.predict_text(user_input)

In [ ]:
app = EmotionAI(base_path)

#Response Generator

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

The response generator uses the FLAN-T5 model (google/flan-t5-base), which is a sequence-to-sequence transformer designed for instruction-based text generation. It takes the user input along with context (such as detected emotion) and generates a natural language response. FLAN-T5 is chosen because it is lightweight, easy to use, and performs well on conversational tasks without requiring large computational resources. This helps the chatbot produce more human-like and context-aware replies.


In [ ]:
def adjust_emotion(user_input, predicted_emotion):
    text = user_input.lower()

    if any(word in text for word in ["happy", "won", "selected", "achieved", "success"]):
        return "happy"

    if any(word in text for word in ["sad", "lonely", "bad", "tired"]):
        return "sad"

    if any(word in text for word in ["angry", "frustrated", "hate"]):
        return "angry"

    return predicted_emotion

In [ ]:
def adjust_emotion(user_input, predicted_emotion):
    text = user_input.lower()

    # negative context first (priority)
    if any(w in text for w in ["tired", "exhausted", "burnt", "drained", "stress", "overwhelmed"]):
        return "sad"

    #  positive context
    if any(w in text for w in ["job", "selected", "won", "achieved", "success"]):
        return "happy"

    #  anger
    if any(w in text for w in ["angry", "frustrated", "hate"]):
        return "angry"

    return predicted_emotion

In [ ]:
def adjust_emotion(user_input, predicted_emotion):
    text = user_input.lower()

    #  NEGATIVE FIRST (highest priority)
    if any(w in text for w in ["tired", "exhausted", "burnt", "drained", "stress", "overwhelmed"]):
        return "sad"

    #  anger
    if any(w in text for w in ["angry", "frustrated", "hate"]):
        return "angry"

    # positive AFTER negative check
    if any(w in text for w in ["job", "selected", "won", "achieved", "success"]):
        return "happy"

    return predicted_emotion

In [ ]:

def adjust_emotion(user_input, predicted_emotion):
    text = user_input.lower()

    # negative first
    if any(w in text for w in ["tired", "exhausted", "burnt", "drained", "stress", "overwhelmed"]):
        return "sad"

    #  anger
    if any(w in text for w in ["angry", "frustrated", "hate"]):
        return "angry"

    #  positive
    if any(w in text for w in ["job", "selected", "won", "achieved", "success", "happy"]):
        return "happy"

    return predicted_emotion

In [ ]:
def build_prompt(user_input, emotion):
    return f"""
You are a helpful and emotionally intelligent assistant.

The user feels {emotion}.
Respond clearly and naturally.

Rules:
- Keep it short (1 sentence)
- Be supportive
- Do NOT repeat the user
- Do NOT generate random or nonsense text

User: {user_input}
Assistant:
"""

The build_prompt function creates a structured input for the response generation model by combining the user’s message with the detected emotion. It provides clear instructions to guide the model to generate short, meaningful, and empathetic responses. Rules such as avoiding repetition and keeping the reply concise help improve the quality and relevance of the output. This prompt engineering step ensures more controlled and human-like responses from the model.


In [ ]:
import random

def generate_response(user_input, emotions):

    text = user_input.lower()
    emotion = adjust_emotion(user_input, emotions[0][0])

    #  polite responses
    if "thank you" in text or "thanks" in text:
        return "You're welcome 😊 I'm here if you want to talk more."

    # greetings
    if text in ["hi", "hello", "hey"]:
        return "Hey there! How are you feeling today?"

    #  context-aware responses

    # job + negative
    if "job" in text and any(w in text for w in ["tired", "exhausted", "stress", "overwhelmed"]):
        return "That sounds really exhausting… work can get overwhelming. What part of it is draining you the most?"

    # job + positive
    if "job" in text:
        return "That's amazing! Congratulations 🎉 What kind of job is it?"

    # lonely
    if "lonely" in text:
        return "I'm really sorry you're feeling lonely. Do you want to talk about it?"

    # tired general
    if "tired" in text:
        return "That sounds exhausting… do you want to share what's been draining you?"

    # 💬 emotion-based fallback
    responses = {
        "happy": [
            "That's great to hear! What made you feel this way?",
            "I love that 😊 Tell me more!",
        ],
        "sad": [
            "I'm really sorry you're feeling this way. Want to talk about it?",
            "That sounds tough… I'm here for you.",
        ],
        "angry": [
            "That sounds frustrating. What happened?",
            "I get why you'd feel that way. Want to talk it out?",
        ],
        "neutral": [
            "I see. Tell me more about it.",
            "Got it. What’s on your mind?",
        ]
    }

    return random.choice(responses.get(emotion, ["I'm here for you."]))


The generate_response function creates chatbot replies using a combination of context-aware rules and emotion-based logic. It first adjusts the detected emotion and then checks for specific keywords (such as greetings, job-related phrases, or emotional states) to provide more relevant responses. If no specific condition is matched, it falls back to predefined responses based on the detected emotion. This hybrid approach ensures the chatbot produces natural, meaningful, and context-sensitive replies without relying entirely on a language model.


In [ ]:
def run_chatbot():
    print("🤖 Emotion AI Chatbot (type 'exit' to quit)\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == "exit":
            print("Bot: Bye! 👋")
            break

        # 🔹 detect emotion
        emotions = app.predict(user_input)

        # 🔹 adjust emotion
        emotion = adjust_emotion(user_input, emotions[0][0])

        print("\nDetected Emotion:", emotion)

        try:
            # 🖼 image input
            if user_input.lower().endswith(('.jpg', '.png', '.jpeg')):
                response = f"It looks like you're feeling {emotion} in this image. Want to tell me what’s going on?"

            # 💬 text input
            else:
                response = generate_response(user_input, emotions)

        except Exception as e:
            print("⚠️ Error:", e)
            response = "I'm here for you. Tell me more."

        print("Bot:", response, "\n")

In [ ]:
run_chatbot()

#Deployment

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def chat_interface(message, history):

    emotions = app.predict(message)
    emotion = adjust_emotion(message, emotions[0][0])

    if message.lower().endswith(('.jpg', '.png', '.jpeg')):
        response = f"It looks like you're feeling {emotion} in this image. Want to tell me what’s going on?"
    else:
        response = generate_response(message, emotions)

    reply = f"🧠 Emotion: {emotion}\n\n💬 {response}"

    history.append((message, reply))
    return "", history


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("## 🤖 Emotion AI Chatbot")
    gr.Markdown("Talk to a chatbot that understands your emotions")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Type your message...", scale=4)
    clear = gr.Button("Clear Chat")

    msg.submit(chat_interface, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch()

In [ ]:
import gradio as gr

def chat_interface(message, image, history):

    # 🖼 If image is uploaded
    if image is not None:
        emotions = app.predict(image)
        emotion = emotions[0][0]

        response = f"It looks like you're feeling {emotion} in this image. Want to tell me what’s going on?"

        user_display = "📷 Image uploaded"

    # 💬 Text input
    else:
        emotions = app.predict(message)
        emotion = adjust_emotion(message, emotions[0][0])
        response = generate_response(message, emotions)

        user_display = message

    reply = f"🧠 Emotion: {emotion}\n\n💬 {response}"

    history.append((user_display, reply))

    return "", None, history


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="purple"
    )
) as demo:

    gr.Markdown("""
    # 🤖 Emotion AI Chatbot
    ### Talk freely — I understand your emotions
    """)

    chatbot = gr.Chatbot(height=400)

    with gr.Row():
        msg = gr.Textbox(placeholder="Type your message...", scale=4)
        image = gr.Image(type="filepath", label="Upload Image")

    with gr.Row():
        send = gr.Button("Send")
        clear = gr.Button("Clear Chat")

    # 🔁 send message
    send.click(chat_interface, [msg, image, chatbot], [msg, image, chatbot])
    msg.submit(chat_interface, [msg, image, chatbot], [msg, image, chatbot])

    # 🧹 clear chat
    clear.click(lambda: ([], ""), None, [chatbot, msg], queue=False)

demo.launch()

In [ ]:
%%writefile app.py
import gradio as gr
import random
import torch
import cv2
from PIL import Image
from torchvision import transforms, models
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json

BASE_PATH = "./"

with open(f"{BASE_PATH}/labels/text_labels.json") as f:
    text_labels = json.load(f)

with open(f"{BASE_PATH}/labels/image_labels.json") as f:
    image_labels = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(f"{BASE_PATH}/text_model")
text_model = AutoModelForSequenceClassification.from_pretrained(f"{BASE_PATH}/text_model")
text_model.eval()

image_model = models.resnet18()
image_model.fc = torch.nn.Linear(image_model.fc.in_features, len(image_labels))
image_model.load_state_dict(torch.load(f"{BASE_PATH}/image_model/resnet_model.pth", map_location="cpu"))
image_model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def map_to_fer(label):
    if label in ['anger', 'annoyance']:
        return 'angry'
    elif label in ['joy', 'amusement', 'love']:
        return 'happy'
    elif label in ['sadness', 'grief']:
        return 'sad'
    else:
        return 'neutral'

def predict_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = text_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]
    idx = torch.argmax(probs).item()
    return map_to_fer(text_labels[idx])

def predict_image(image_path):
    image_cv = cv2.imread(image_path)
    gray = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    if len(faces) > 0:
        (x, y, w, h) = faces[0]
        face = image_cv[y:y+h, x:x+w]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(face)
    else:
        image = Image.open(image_path).convert("RGB")

    image = transform(image).unsqueeze(0)

    with torch.no_grad():
        outputs = image_model(image)
        probs = torch.softmax(outputs, dim=1)[0]

    idx = torch.argmax(probs).item()
    return image_labels[idx]

def adjust_emotion(user_input, predicted):
    text = user_input.lower()

    if "exhaust" in text or "tired" in text:
        return "sad"
    if "angry" in text:
        return "angry"
    if "job" in text or "happy" in text:
        return "happy"

    return predicted

def generate_response(user_input, emotion):
    if "thank" in user_input.lower():
        return "You're welcome 😊"

    if emotion == "happy":
        return "That's amazing! Tell me more!"
    elif emotion == "sad":
        return "I'm sorry you're feeling this way. Want to talk about it?"
    elif emotion == "angry":
        return "That sounds frustrating. What happened?"
    else:
        return "I see. Tell me more."

def chat(message, image, history):
    if image:
        emotion = predict_image(image)
        response = f"You look {emotion}. Tell me more."
        user = "📷 Image"
    else:
        pred = predict_text(message)
        emotion = adjust_emotion(message, pred)
        response = generate_response(message, emotion)
        user = message

    history.append((user, f"{emotion} → {response}"))
    return "", None, history

with gr.Blocks() as demo:
    gr.Markdown("# Emotion AI Chatbot")

    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    img = gr.Image(type="filepath")

    btn = gr.Button("Send")

    btn.click(chat, [msg, img, chatbot], [msg, img, chatbot])

demo.launch()

In [ ]:
%%writefile requirements.txt
torch
transformers
gradio
opencv-python-headless
Pillow
torchvision